In [1]:
!pip install -q transformers==4.46.3 tokenizers==0.20.3 addict easydict einops kagglehub
import transformers
print("transformers:", transformers.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.8 MB/s eta 0:00:00
transformers: 4.46.3


In [2]:
import torch, kagglehub
from pathlib import Path

print("GPU:", torch.cuda.get_device_name(0))   # Tesla T4 bo'lishi shart
DATA = Path(kagglehub.dataset_download("sarvarbekerniyazov/ocripple-images"))
n = sum(1 for _ in DATA.rglob("*.png"))
print("DATA:", DATA, "| pngs:", n)

GPU: Tesla T4
DATA: /kaggle/input/datasets/sarvarbekerniyazov/ocripple-images | pngs: 522


In [3]:
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "baidu/Unlimited-OCR"
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_ID, trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    _attn_implementation="eager").eval()
print("model loaded")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

modeling_unlimitedocr.py: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/baidu/Unlimited-OCR:
- conversation.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/baidu/Unlimited-OCR:
- modeling_unlimitedocr.py
- deepencoder.py
- modeling_deepseekv2.py
- conversation.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-000001.safetensors:   0%|          | 0.00/6.67G [00:00<?, ?B/s]

Some weights of UnlimitedOCRForCausalLM were not initialized from the model checkpoint at baidu/Unlimited-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model loaded


In [4]:
torch._C._jit_override_can_fuse_on_gpu(False)
torch._C._jit_set_texpr_fuser_enabled(False)
torch._C._jit_set_profiling_executor(False)
torch._C._jit_set_profiling_mode(False)
print("JIT fusion disabled")

JIT fusion disabled


In [5]:
import re

PROMPT = "<image>\nFree OCR."
TMP = "/kaggle/working/uo_tmp"

def clean_unlimited_output(text):
    return re.sub(r'<\|det\|>[^<]*<\|/det\|>', '', text)

def unlimited_page(img_path):
    res = model.infer(tok, prompt=PROMPT, image_file=str(img_path),
                      output_path=TMP,
                      base_size=1024, image_size=640, crop_mode=True,
                      save_results=False, eval_mode=True)
    raw = res if isinstance(res, str) else str(res)
    return clean_unlimited_output(raw)

In [6]:
test_img = next((DATA/"arxiv_scans"/"clean").glob("*.png"))
out = unlimited_page(test_img)
print(repr(out[:400]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48


'\nFigure 2: BERT input representation. The input embeddings are the sum of the token embeddings, the segmentation embeddings and the position embeddings.\nThe NSP task is closely related to representation-learning objectives used in Jernite et al. (2017) and Logeswaran and Lee (2018). However, in prior work, only sentence embeddings are transferred to down-stream tasks, where BERT transfers all para'


In [7]:
import time, json
from tqdm.auto import tqdm

OUT = Path("/kaggle/working/ocr_outputs")

def collect_images():
    tasks = []
    for lv in ["clean", "light", "medium", "heavy"]:
        for p in sorted((DATA/"arxiv_scans"/lv).glob("*.png")):
            tasks.append((p, f"arxiv/{lv}/{p.stem}"))
    for p in sorted((DATA/"mpdocvqa"/"pages").glob("*.png")):
        tasks.append((p, f"mpdocvqa/{p.stem}"))
    assert tasks, "0 images found!"
    return tasks

def run_engine(engine_name, page_fn):
    tasks = collect_images()
    print(f"{engine_name}: {len(tasks)} images")
    eng_dir = OUT/engine_name
    eng_dir.mkdir(parents=True, exist_ok=True)
    timing = []
    for img_path, stem in tqdm(tasks, desc=engine_name):
        out = eng_dir/f"{stem}.txt"
        out.parent.mkdir(parents=True, exist_ok=True)
        if out.exists():
            continue
        t0 = time.perf_counter()
        try:
            text = page_fn(img_path)
        except Exception as e:
            text = ""
            print("ERR", img_path.name, repr(e)[:120])
        timing.append({"image": img_path.name,
                       "sec": round(time.perf_counter()-t0, 3)})
        out.write_text(text or "", encoding="utf-8")
    (eng_dir/"timing.json").write_text(json.dumps(timing))
    print("done:", len(timing))

print("images total:", len(collect_images()))   # 522 kutiladi

images total: 522


In [8]:
run_engine("unlimited_ocr", unlimited_page)

unlimited_ocr: 522 images


unlimited_ocr:   0%|          | 0/522 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


error: image file is truncated
ERR hqgb0228_p4.png OSError('image file is truncated')


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


...

ERR xkbv0228_p5.png OutOfMemoryError('CUDA out of memory. Tried to allocate 616.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 224


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


ERR zrhw0228_p0.png OutOfMemoryError('CUDA out of memory. Tried to allocate 586.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 224


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


done: 522


In [9]:
import shutil
shutil.make_archive("/kaggle/working/unlimited_outputs", "zip",
                    "/kaggle/working/ocr_outputs/unlimited_ocr")

'/kaggle/working/unlimited_outputs.zip'